In [ ]:
import os
import json

from datasets import Dataset, DatasetDict, load_from_disk, load_dataset
from datasets.features import Value, ClassLabel
from datasets.combine import concatenate_datasets

from datasets_utils import dataset_class_encode_column, validate_equal_datasets

# Dataset for image classification

In [ ]:
datasets2concat = {
    "milk10k": "/home/sulcm/datasets/milk10k/milk10k",
    "isic2019": "/home/sulcm/datasets/isic2019/isic2019",
}

CLASS_LABEL_COLUMN = "label"
FEATURE_COLUMN = "image"

SEED = 42

im_cls_metadata = {}
with open("../training/image_classification/configs/metadata.json", "r") as f:
    _ld_metadata = json.load(f)
    if isinstance(_ld_metadata, dict):
        im_cls_metadata.update(**_ld_metadata)
    else:
        im_cls_metadata["additional_metadata"] = _ld_metadata

In [ ]:
valid_columns = [
    CLASS_LABEL_COLUMN,
    FEATURE_COLUMN,
]

ld_datasets = []
label2id = {l: i for i, l in enumerate(im_cls_metadata["labels"])}
for d_name, d_path in datasets2concat.items():
    if os.path.exists(d_path):
        _ds = load_from_disk(d_path)
    else:
        _ds = load_dataset(d_path)

    for split in _ds.keys():
        _ds[split] = _ds[split].remove_columns(
            column_names=[c_name for c_name in _ds[split].column_names if c_name not in valid_columns]
        )
        _ds[split] = _ds[split].add_column(name="original_ds_row_id", column=list(range(len(_ds[split]))))
        _ds[split] = _ds[split].add_column(name="original_ds_name", column=[d_name]*len(_ds[split]))
        _ds[split] = _ds[split].add_column(name="original_ds_split", column=[split]*len(_ds[split]))

    if all([isinstance(s.features[CLASS_LABEL_COLUMN], ClassLabel) for s in _ds.values()]):
        new_ds = _ds.align_labels_with_mapping(label2id, CLASS_LABEL_COLUMN)
    else:
        new_ds = dataset_class_encode_column(
            dataset=_ds,
            column=CLASS_LABEL_COLUMN,
            custom_labels=im_cls_metadata["labels"]
        )

    res, msg = validate_equal_datasets(_ds, new_ds, columns=valid_columns)
    if res:
        print(f"Dataset {d_name} was loaded, processed, and successfully validated. Adding to list ...")
    else:
        raise Value(f"Dataset {d_name} is not valid: {msg}")

    ld_datasets.append(new_ds)

## Balance class distribution between splits
- use stratified sampling (by class label) to create splits => alleviates the problem of Random Sampling in datasets with an imbalanced-class distribution
    - keeps the distribution of classes in each of the train, validation, and test sets preserved
- optionally set seed for reproducibility
- splits are created iterativly
    - Dataset concatenation ("single split")
    - Split dataset into __real__ train part and temporary validation/test part
    - Split the temporarily created part into __final__ validation and test parts

In [ ]:
dset_concat: Dataset = concatenate_datasets([dset_split for dset in ld_datasets for dset_split in dset.values()])

In [ ]:
dset_initial_split = dset_concat.train_test_split(
    test_size=0.2,
    shuffle=True,
    stratify_by_column=CLASS_LABEL_COLUMN,
    seed=SEED
)
dset_initial_split

In [ ]:
dset_val_test_split = dset_initial_split["test"].train_test_split(
    test_size=0.5,
    shuffle=True,
    stratify_by_column=CLASS_LABEL_COLUMN,
    seed=SEED
)
dset_val_test_split

In [ ]:
dset_final = DatasetDict({
    "train": dset_initial_split["train"],
    "validation": dset_val_test_split["train"],
    "test": dset_val_test_split["test"]
})
dset_final

In [ ]:
# dset_final.save_to_disk("/home/sulcm/datasets/MelanoMix")

In [ ]:
melano_mix = load_from_disk("/home/sulcm/datasets/MelanoMix")
melano_mix